# 01 - Exploratory Data Analysis (EDA)

This notebook performs an extensive Exploratory Data Analysis (EDA) on the raw 
insurance claims data. The goal is to understand the data's structure, identify patterns, 
detect anomalies, and prepare it for subsequent modeling and hypothesis testing.

We will cover:

1.Loading the raw data.

2.Initial data overview (shape, head, tail, info, describe).

3.Missing value analysis.

4.Unique value counts for categorical features.

5.Distribution analysis of key numerical features and target variables.

6.Correlation analysis.

The data preparation steps (loading, cleaning, feature engineering, missing data 
handling) will primarily utilize functions from src/data_preparation.py and 
src/feature_engineering.py for consistency with the overall pipeline.

In [ ]:
# --- Setup and Imports ---
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add the project root to the Python path for module imports
script_dir = os.path.dirname(os.path.abspath('')) # Get current notebook directory
project_root = os.path.abspath(os.path.join(script_dir, '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root '{project_root}' to sys.path.")
else:
    print(f"Project root '{project_root}' was already in sys.path.")

# Import modules from src
from src.data_preparation import load_raw_data, _convert_comma_to_dot_and_numeric, handle_missing_data
from src.feature_engineering import engineer_features # Import engineer_features separately
from src.config import (
    RAW_DATA_PATH, TOTAL_CLAIMS_COL, TOTAL_PREMIUM_COL, HAS_CLAIM_COL, MARGIN_COL,
    NUMERICAL_FEATURES_CLASSIFICATION, CATEGORICAL_FEATURES_CLASSIFICATION, DATE_FEATURES_CLASSIFICATION,
    POLICY_ID_COL, DRIVER_GENDER_COL, PROVINCE_COL, POSTAL_CODE_COL
)

# Configure plotting styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 1. Load Raw Data

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
if df_raw.empty:
    print("Failed to load raw data. Please check RAW_DATA_PATH in src/config.py and data file existence.")
else:
    print("Raw data loaded successfully. Displaying first 5 rows:")
    display(df_raw.head())

# 2. Initial Data Cleaning & Feature Derivation for EDA

In [ ]:
# Create a copy for EDA-specific processing to avoid modifying the original loaded df_raw
df_eda = df_raw.copy()
df_eda.columns = df_eda.columns.str.strip()

# Define all columns that are or should be numeric for initial conversion (as per config and data observations)
# This is a superset to catch all numerical columns potentially existing or engineered
potential_numeric_cols = list(set(
    NUMERICAL_FEATURES_CLASSIFICATION + 
    NUMERICAL_FEATURES_REGRESSION + # Include regression features too for comprehensive EDA
    [TOTAL_CLAIMS_COL, TOTAL_PREMIUM_COL, 'CustomValueEstimate', 'CapitalOutstanding',
     'SumInsured', 'CalculatedPremiumPerTerm', MARGIN_COL, HAS_CLAIM_COL,
     'PremiumPerSumInsured', 'VehicleAge_at_Transaction',
     f'{TRANSACTION_MONTH_COL}_Year', f'{TRANSACTION_MONTH_COL}_Month',
     'VehicleIntroDate_Year', 'VehicleIntroDate_Month'
    ],
))
# Filter to only include columns actually present in the DataFrame for conversion
potential_numeric_cols_in_df = [col for col in potential_numeric_cols if col in df_eda.columns]

# Perform comma-to-dot and numeric conversion for all relevant numerical columns
df_eda = _convert_comma_to_dot_and_numeric(df_eda, potential_numeric_cols_in_df)

# Calculate HasClaim and Margin (needed for EDA on targets). These are now handled in data_preparation.py
# for a fuller pipeline, but can be explicitly shown here for clarity in EDA.
if TOTAL_CLAIMS_COL in df_eda.columns:
    df_eda[HAS_CLAIM_COL] = (pd.to_numeric(df_eda[TOTAL_CLAIMS_COL], errors='coerce').fillna(0) > 0).astype(int)
else:
    df_eda[HAS_CLAIM_COL] = 0

if TOTAL_PREMIUM_COL in df_eda.columns and TOTAL_CLAIMS_COL in df_eda.columns:
    df_eda[MARGIN_COL] = pd.to_numeric(df_eda[TOTAL_PREMIUM_COL], errors='coerce').fillna(0) - pd.to_numeric(df_eda[TOTAL_CLAIMS_COL], errors='coerce').fillna(0)
else:
    df_eda[MARGIN_COL] = 0.0

# Ensure non-negativity for relevant financial/count columns after derivation
for col in potential_numeric_cols_in_df + [MARGIN_COL]:
    if col in df_eda.columns and pd.api.types.is_numeric_dtype(df_eda[col]):
        df_eda[col] = df_eda[col].clip(lower=0)

# Apply feature engineering for derived date features and others
df_eda_engineered = engineer_features(df_eda, DATE_FEATURES_CLASSIFICATION)

# Finally, handle missing data using the robust function from data_preparation on all current columns
# Identify all numerical and categorical columns present after engineering for comprehensive imputation
all_current_numeric_cols = [col for col in df_eda_engineered.select_dtypes(include=np.number).columns if col not in [HAS_CLAIM_COL]]
all_current_categorical_cols = [col for col in df_eda_engineered.select_dtypes(include='object').columns]

df_final_eda = handle_missing_data(df_eda_engineered, 
                                   all_current_numeric_cols, 
                                   all_current_categorical_cols)

print("Data cleaned and features derived for EDA. Displaying info and first rows:")
df_final_eda.info()
display(df_final_eda.head())

# 3. Data Overview

In [ ]:
print("Shape:", df_final_eda.shape)
print("\nColumn Information (dtypes and non-null counts):")
df_final_eda.info()
print("\nDescriptive Statistics for Numerical Columns:")
display(df_final_eda.describe().T)

# 4. Missing Values Analysis

In [ ]:
missing_data = df_final_eda.isnull().sum()
missing_percent = (df_final_eda.isnull().sum() / len(df_final_eda)) * 100
missing_df = pd.DataFrame({'Total Missing': missing_data, 'Percentage (%)': missing_percent})
missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by='Total Missing', ascending=False)
print("Missing Values Summary (after cleaning and imputation):")
if missing_df.empty:
    print("No missing values found in the DataFrame.")
else:
    display(missing_df)
    # Verify that there are truly no NaNs after the final imputation step for critical columns
    print("\nVerifying NaNs in sample columns after imputation:")
    sample_cols_to_verify = [col for col in ['TotalPremium', 'SumInsured', 'PolicyType', 'DriverGender', 'Age'] if col in df_final_eda.columns]
    for col in sample_cols_to_verify:
        print(f"  {col} NaN count: {df_final_eda[col].isnull().sum()}")

# 5. Unique Values for Categorical Columns

In [ ]:
categorical_cols = df_final_eda.select_dtypes(include='object').columns
for col in categorical_cols:
    print(f"\n--- Column: {col} ---")
    print(f"Unique values: {df_final_eda[col].nunique()}")
    # Display top 10 categories and their counts/percentages
    top_categories = df_final_eda[col].value_counts(dropna=False)
    print("Top 10 categories:")
    display(top_categories.head(10))
    print("Top 10 categories (%):")
    display(top_categories.head(10) / len(df_final_eda) * 100)

# 6. Distribution Analysis of Target Variables

In [ ]:
# 6.1. Claim Frequency (HasClaim)
if HAS_CLAIM_COL in df_final_eda.columns:
    print(f"\n{HAS_CLAIM_COL} (Claim Frequency) Distribution:")
    claim_counts = df_final_eda[HAS_CLAIM_COL].value_counts(normalize=False)
    claim_percentages = df_final_eda[HAS_CLAIM_COL].value_counts(normalize=True) * 100
    claim_dist = pd.DataFrame({'Count': claim_counts, 'Percentage (%)': claim_percentages})
    display(claim_dist)

    plt.figure(figsize=(7, 5))
    sns.countplot(x=HAS_CLAIM_COL, data=df_final_eda, palette='viridis')
    plt.title(f'Distribution of {HAS_CLAIM_COL} (0: No Claim, 1: Has Claim)')
    plt.xlabel('Has Claim')
    plt.ylabel('Number of Policies')
    plt.show()
else:
    print(f"'{HAS_CLAIM_COL}' column not found for target analysis.")

# 6.2. Claim Severity (TotalClaims conditional on HasClaim > 0)
if TOTAL_CLAIMS_COL in df_final_eda.columns and HAS_CLAIM_COL in df_final_eda.columns:
    print(f"\n{TOTAL_CLAIMS_COL} (Claim Severity) Distribution (for policies with claims):")
    claims_only_df = df_final_eda[df_final_eda[HAS_CLAIM_COL] == 1].copy()
    if not claims_only_df.empty:
        print(claims_only_df[TOTAL_CLAIMS_COL].describe())
        plt.figure(figsize=(10, 6))
        sns.histplot(claims_only_df[TOTAL_CLAIMS_COL], bins=50, kde=True, color='red')
        plt.title(f'Distribution of {TOTAL_CLAIMS_COL} (Conditional on Claim)')
        plt.xlabel('Claim Amount')
        plt.ylabel('Frequency')
        plt.yscale('log') # Log scale often useful for skewed financial data
        plt.grid(True, which=\"both\", ls=\"--\", c='0.7')
        plt.show()
    else:
        print("No policies with claims found for Claim Severity analysis.")
else:
    print(f"'{TOTAL_CLAIMS_COL}' or '{HAS_CLAIM_COL}' columns not found for Claim Severity analysis.")

# 6.3. Margin Distribution
if MARGIN_COL in df_final_eda.columns:
    print(f"\n{MARGIN_COL} Distribution:")
    print(df_final_eda[MARGIN_COL].describe())
    plt.figure(figsize=(10, 6))
    sns.histplot(df_final_eda[MARGIN_COL], bins=50, kde=True, color='green')
    plt.title(f'Distribution of {MARGIN_COL}')
    plt.xlabel('Margin (Premium - Claims)')
    plt.ylabel('Frequency')
    plt.grid(True, which=\"both\", ls=\"--\", c='0.7')
    plt.show()
else:
    print(f"'{MARGIN_COL}' column not found for Margin analysis.")

# 7. Feature Distributions (Sampled Visualizations)

In [ ]:
print("\n7.1. Numerical Feature Distributions:")
num_cols_to_plot = [col for col in ['TotalPremium', 'SumInsured', 'Age', 'VehicleAge', 'PremiumPerSumInsured', 'VehicleAge_at_Transaction'] if col in df_final_eda.columns]
for col in num_cols_to_plot:
    if not df_final_eda[col].empty and pd.api.types.is_numeric_dtype(df_final_eda[col]):
        plt.figure(figsize=(8, 5))
        sns.histplot(df_final_eda[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.show()
    else:
        print(f"Warning: '{col}' is not suitable for numerical distribution plot or not found.")

print("\n7.2. Categorical Feature Distributions:")
cat_cols_to_plot = [col for col in ['PolicyType', DRIVER_GENDER_COL, 'VehicleType', PROVINCE_COL, 'TermFrequency', 'AlarmImmobiliser', 'NewVehicle'] if col in df_final_eda.columns]
for col in cat_cols_to_plot:
    if not df_final_eda[col].empty and (df_final_eda[col].dtype == 'object' or pd.api.types.is_string_dtype(df_final_eda[col])):
        plt.figure(figsize=(8, 5))
        sns.countplot(y=col, data=df_final_eda, order=df_final_eda[col].value_counts().index, palette='pastel')
        plt.title(f'Distribution of {col}')
        plt.xlabel('Count')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Warning: '{col}' is not suitable for categorical distribution plot or not found.")

# 8. Correlation Analysis

In [ ]:
print("\n8.1. Correlation Matrix of Numerical Features:")
# Select only numerical columns for correlation calculation
numeric_df_for_corr = df_final_eda.select_dtypes(include=np.number)
if not numeric_df_for_corr.empty:
    plt.figure(figsize=(14, 12))
    sns.heatmap(numeric_df_for_corr.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, annot_kws={"size": 8})
    plt.title('Correlation Matrix of Numerical Features (after cleaning and engineering)')
    plt.show()
else:
    print("No numerical columns found for correlation matrix after processing.")

print("\n8.2. Correlation with Target Variables:")
if HAS_CLAIM_COL in df_final_eda.columns and not df_final_eda[HAS_CLAIM_COL].empty:
    # Correlation of numerical features with 'HasClaim' (point-biserial correlation)
    # For a binary target, correlation is equivalent to point-biserial correlation.
    # We'll use the .corr() method which computes Pearson correlation by default, which is appropriate.
    numeric_features_for_target_corr = [col for col in numeric_df_for_corr.columns if col != HAS_CLAIM_COL]
    if numeric_features_for_target_corr:
        corr_with_has_claim = df_final_eda[numeric_features_for_target_corr + [HAS_CLAIM_COL]].corr().loc[:, HAS_CLAIM_COL].drop(HAS_CLAIM_COL).sort_values(ascending=False)
        print(f"\nCorrelation with '{HAS_CLAIM_COL}':")
        display(corr_with_has_claim)
        
        plt.figure(figsize=(8, 6))
        corr_with_has_claim.plot(kind='barh', color='lightcoral')
        plt.title(f'Correlation of Numerical Features with {HAS_CLAIM_COL}')
        plt.xlabel('Correlation Coefficient')
        plt.ylabel('Feature')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No numerical features available for correlation with {HAS_CLAIM_COL}.")
else:
    print(f"'{HAS_CLAIM_COL}' column not found for correlation analysis.")

if TOTAL_CLAIMS_COL in df_final_eda.columns and HAS_CLAIM_COL in df_final_eda.columns:
    claims_only_numeric_df = df_final_eda[df_final_eda[HAS_CLAIM_COL] == 1].select_dtypes(include=np.number)
    if not claims_only_numeric_df.empty and TOTAL_CLAIMS_COL in claims_only_numeric_df.columns:
        numeric_features_for_severity_corr = [col for col in claims_only_numeric_df.columns if col != TOTAL_CLAIMS_COL and col != HAS_CLAIM_COL]
        if numeric_features_for_severity_corr:
            corr_with_severity = claims_only_numeric_df[numeric_features_for_severity_corr + [TOTAL_CLAIMS_COL]].corr().loc[:, TOTAL_CLAIMS_COL].drop(TOTAL_CLAIMS_COL).sort_values(ascending=False)
            print(f"\nCorrelation with '{TOTAL_CLAIMS_COL}' (Conditional on Claim):")
            display(corr_with_severity)
            
            plt.figure(figsize=(8, 6))
            corr_with_severity.plot(kind='barh', color='lightgreen')
            plt.title(f'Correlation of Numerical Features with {TOTAL_CLAIMS_COL} (Conditional on Claim)')
            plt.xlabel('Correlation Coefficient')
            plt.ylabel('Feature')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No numerical features available for correlation with {TOTAL_CLAIMS_COL} (conditional).")
    else:
        print(f"No numerical columns or no claims found for {TOTAL_CLAIMS_COL} correlation analysis.")

if MARGIN_COL in df_final_eda.columns and not df_final_eda[MARGIN_COL].empty:
    numeric_features_for_margin_corr = [col for col in numeric_df_for_corr.columns if col != MARGIN_COL and col != HAS_CLAIM_COL and col != TOTAL_CLAIMS_COL]
    if numeric_features_for_margin_corr:
        corr_with_margin = df_final_eda[numeric_features_for_margin_corr + [MARGIN_COL]].corr().loc[:, MARGIN_COL].drop(MARGIN_COL).sort_values(ascending=False)
        print(f"\nCorrelation with '{MARGIN_COL}':")
        display(corr_with_margin)
        
        plt.figure(figsize=(8, 6))
        corr_with_margin.plot(kind='barh', color='lightgray')
        plt.title(f'Correlation of Numerical Features with {MARGIN_COL}')
        plt.xlabel('Correlation Coefficient')
        plt.ylabel('Feature')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No numerical features available for correlation with {MARGIN_COL}.")
else:
    print(f"'{MARGIN_COL}' column not found for correlation analysis.")

# 9. Conclusion and Next Steps

This EDA has provided insights into the data's quality, distributions, and initial 
relationships between features and targets. Key observations include:

Data Quality: Our preprocessing steps (delimiter correction, numeric coercion, 
    missing value imputation) have significantly improved data quality, though some 
    columns still required careful handling.

Class Imbalance: The HasClaim target variable shows a significant class 
    imbalance, which will need to be addressed during classification model training 
    (e.g., via class_weight or resampling).

Skewed Distributions: Financial columns (TotalPremium, TotalClaims, SumInsured, Margin) 
    exhibit heavily skewed distributions, which is typical for such data. This may influence 
    the choice of regression models or necessitate transformations.

Feature Relationships: Initial correlations provide hints about which features might be 
    strong predictors for claim probability, severity, and margin. Categorical features 
    have diverse distributions, indicating their potential importance.

Next Steps:

Proceed to 02_hypothesis_testing.ipynb to statistically validate key hypotheses about risk drivers.
Proceed to 03_model_development.ipynb to build and evaluate predictive models for claim 
probability and severity, leveraging the insights from EDA and the refined data preparation pipeline.